In [7]:
# Header for the notebook
from datetime import datetime
from IPython.display import display, Markdown

# Get the current date
title = "Python notebook to analyse the HRV in Borderline Personality Disorder"
current_date = datetime.now().strftime("%d %B %Y, %H:%M:%S")
authors = "Estelle Thomas (with Copilot & Claude)"

# Insert the date into the notebook
display(Markdown(f"# {title}"))
display(Markdown(f"{current_date}"))
display(Markdown(f"By {authors}"))

# Python notebook to analyse the HRV in Borderline Personality Disorder

12 May 2026, 07:17:15

By Estelle Thomas (with Copilot & Claude)

# RMSSD calculation from RR interval recordings
*Master 2 REHAB, Faculty of Medicine, Montpellier*

This notebook computes the Root Mean Square of Successive Differences (RMSSD) 
from raw RR interval recordings for each participant (BPD and healthy controls).  
The output is an Excel file used as input for the statistical analysis in R.

In [8]:
# Installer openpyxl pour pouvoir exporter le dataframe dans un fichier excel
#!pip install openpyxl
#!pip install neurokit2
# Comment lines if packages are already installed, please

# 1. Import necessary libraries
import pandas as pd
import os
import neurokit2 as nk
import numpy as np



## 2. RMSSD computation from RR intervals

For each participant, the `.txt` file contains raw RR intervals (in milliseconds), 
one value per line.  

The pipeline applies the following steps:  
1. **Physiological filter**: RR intervals outside the 300–2000 ms range are removed,  
   as they correspond to physiologically implausible values (Task Force ESC/NASPE, 1996; Kamath & Fallen, 1995).  
2. **Artifact filter**: successive intervals with a relative change greater than 20% are removed  
   to eliminate movement artifacts.  
3. **RMSSD calculation**: the cleaned intervals are converted to peaks and passed to  
   `nk.hrv_time()` (NeuroKit2, Makowski et al., 2021, Pham et al., 2021).  

**Input**: `XX_BPD.txt` and `XX_CON.txt` files in `data/BPD` and `data/CONTROL` folders.  
**Output**: `HRV_results.xlsx` saved in the `results` folder, containing one row per participant  
with columns `ID`, `Group`, and `HRV_RMSSD`.

In [9]:
# Folder with RR intervals in data/BPD and data/CONTROL
data_dirs = ["data/BPD", "data/CONTROL"]
 
results = []

# Process the RR interval files for each folder (BPD and CONTROL)
for data_dir in data_dirs:
 
    for filename in sorted(os.listdir(data_dir)):
        if not filename.endswith(".txt"):
            continue
 
        # Load the RR intervals
        rr = np.loadtxt(os.path.join(data_dir, filename))

        # Clean the RR intervals (Task Force ESC/NASPE, 1996 ; Kamath & Fallen, 1995)
        # Remove artifacts beats (physiological implausible intervals) IBI
        rr_clean = rr[(rr > 300) & (rr < 2000)]
        pct_change = np.abs(np.diff(rr_clean) / rr_clean[:-1]) * 100
        mask = np.concatenate([[True], pct_change <= 20])
        rr_clean = rr_clean[mask]

        # Convert the RR intervals to peaks (heartbeats)
        peaks = nk.intervals_to_peaks(rr_clean, sampling_rate=1000)

        # Calculate HRV indices (RMSSD)
        hrv = nk.hrv_time(peaks, sampling_rate=1000)

        # Extract ID and Group from the filename
        participant_id = filename.split('_')[0]  # Subject ID
        group = filename.split('_')[1].split('.')[0]  # Group (BPD or CONTROL)

        results.append({
            'ID': participant_id,
            'Group': group,
            'HRV_RMSSD': hrv['HRV_RMSSD'].values[0],
        })
       


## 3. Results

The table below shows the RMSSD value computed for each participant.  
The DataFrame is then exported to `results/HRV_results.xlsx` for use in the R statistical analysis.

In [10]:
# Create and export a DataFrame from the results
hrv_df = pd.DataFrame(results)
print(hrv_df)

hrv_df.to_excel("results/HRV_results.xlsx", index=False)

    ID Group   HRV_RMSSD
0   10   BPD   88.115652
1   11   BPD   48.957069
2   12   BPD    8.625192
3   13   BPD   27.252732
4   14   BPD   12.454390
5   15   BPD   24.861691
6   16   BPD    8.860358
7   17   BPD   18.327676
8   18   BPD   21.970706
9   19   BPD   84.751512
10   1   BPD   10.953634
11  20   BPD   40.268262
12  21   BPD   17.670730
13  22   BPD   13.245632
14  23   BPD   68.692331
15  24   BPD    8.652065
16  25   BPD   43.187697
17  26   BPD   18.377157
18  27   BPD   34.759791
19  28   BPD   30.806670
20  29   BPD   16.261135
21   2   BPD   17.314829
22  30   BPD   33.769537
23  31   BPD   19.908611
24  32   BPD   12.591342
25   3   BPD   37.902549
26   4   BPD   38.267364
27   5   BPD   26.363223
28   6   BPD   35.552120
29   7   BPD    8.671538
30   8   BPD   32.704740
31   9   BPD   20.202783
32  33   CON   37.784950
33  34   CON   45.068058
34  35   CON   12.231409
35  36   CON   91.536332
36  37   CON   14.357911
37  38   CON   58.021698
38  39   CON   29.770737


# Reference : 
Makowski, D., Pham, T., Lau, Z. J., Brammer, J. C., Lespinasse, F., Pham, H., Schölzel, C., & Chen, S. H. A. (2021). NeuroKit2: A Python Toolbox for Neurophysiological Signal Processing. Behavior Research Methods, 53(4), 1689–1696. https://doi.org/10.3758/s13428-020-01516-y

Task Force of the European Society of Cardiology and the North American Society of Pacing and Electrophysiology. (1996). Heart rate variability: Standards of measurement, physiological interpretation and clinical use. Circulation, 93(5), 1043–1065. https://doi.org/10.1161/01.CIR.93.5.1043

Kamath, M. V., & Fallen, E. L. (1995). Correction of the heart rate variability signal for ectopics and missing beats. In M. Malik & A. J. Camm (Eds.), Heart Rate Variability (pp. 75–85). Futura Publishing Company.


Pham, T., Lau, Z. J., Chen, S. H. A., & Makowski, D. (2021). Heart rate variability in psychology: A review of HRV indices and an analysis tool for the meta-analysis of heart rate variability in sleep and memory. Sensors, 21(12), 3998. https://doi.org/10.3390/s21123998





In [11]:
# https://nbconvert.readthedocs.io/en/latest/removing_cells.html
# https://github.com/msm1089/ipynbname/issues/17#issuecomment-1293269863


from traitlets.config import Config
from nbconvert.exporters import HTMLExporter
from nbconvert.preprocessors import TagRemovePreprocessor
from IPython.core.getipython import get_ipython
import os


def get_notebook_name():
    """
    Get the current notebook name (without extension).
    """
    ip = get_ipython()
    path = None
    if "__vsc_ipynb_file__" in ip.user_ns:  # type: ignore
        path = ip.user_ns["__vsc_ipynb_file__"] # type: ignore
        # split the path to get only the file name without extension

    return os.path.splitext(path)[0]  # type: ignore


# Get the notebook name
notebook_file_name = get_notebook_name()

# Switch to inline backend for HTML export (interactive backends don't work in static HTML)
get_ipython().run_line_magic("matplotlib", "inline")  # type: ignore
print("Switched to 'inline' backend for HTML export")


# Setup config
c = Config()

# Configure tag removal - be sure to tag your cells to remove  using the
# words remove_cell to remove cells. You can also modify the code to use
# a different tag word
c.TagRemovePreprocessor.remove_cell_tags = ("remove",)
c.TagRemovePreprocessor.remove_all_outputs_tags = ("remove_output",)
c.TagRemovePreprocessor.remove_input_tags = ("hide",)
c.TagRemovePreprocessor.enabled = True
c.HTMLExporter.preprocessors = ["nbconvert.preprocessors.TagRemovePreprocessor"]

# ensure the graphics are included in the html
c.HTMLExporter.embed_images = True
# do not show the input code cells (distracts from the output)
c.HTMLExporter.exclude_output_prompt = True
c.HTMLExporter.exclude_input_prompt = True

# Configure the exporter
exporter = HTMLExporter(config=c)
exporter.register_preprocessor(TagRemovePreprocessor(config=c), True)


# run our exporter - returns a tuple - first element with html,
# second with notebook metadata
output = HTMLExporter(config=c).from_filename(notebook_file_name + ".ipynb")

# Write to output html file
with open(notebook_file_name + ".html", "w") as f:
    f.write(output[0])

# open the file with the operating system's default program
# handle spaces in filename with quotes
try:
    if os.name == "posix":
        if os.uname().sysname == "Darwin":
            # macOS - use single quotes for spaces
            os.system(f"open '{notebook_file_name}.html'")
        else:
            # Linux - use single quotes for spaces
            os.system(f"xdg-open '{notebook_file_name}.html'")
    elif os.name == "nt":
        # Windows - use double quotes for empty title + for file name with spaces
        os.system(f'start "" "{notebook_file_name}.html"')
    else:
        print("Unsupported OS")
except Exception as e:
    print("Error opening file: ", e)


Switched to 'inline' backend for HTML export
